# 00 - 系统清单 (System Inventory)

**目的**: 自动探测并记录当前运行的所有技术组件，形成可追溯的技术栈基线文档  
**输出**: 每个服务的版本 / 端口 / 配置 / 连通性状态  
**建议**: 每次部署后运行一次，留存快照。作为其他 Notebook 的前置条件检查。

**当前技术栈（设计层）**:
```
User → Nginx(80) → xiaocai-api(8001/GraphQL) → xiaocai-ai-engine(8002/LangGraph)
                                                        ↓
                              ┌─────────────────────────────────────────┐
                              │  LangGraph Workflow                     │
                              │  ├─ Intent Detection  → APIPro/Ollama   │
                              │  ├─ Session           → PostgreSQL       │
                              │  ├─ Embedding         → sentence-trans   │
                              │  │     ↓ cache        → Redis           │
                              │  ├─ Vector Search     → Qdrant          │
                              │  └─ Main LLM          → APIPro/Ollama   │
                              └─────────────────────────────────────────┘
```

## 0. 配置（修改此处）

In [ ]:
import os
from datetime import datetime

# =============================================
# 服务端点配置
# =============================================
SERVICES = {
    "nginx":      {"url": "http://localhost:80",    "health_path": "/health"},
    "api":        {"url": "http://localhost:8001",  "health_path": "/health"},
    "ai_engine":  {"url": "http://localhost:8002",  "health_path": "/health"},
    "qdrant":     {"url": "http://localhost:6333",  "health_path": "/healthz"},
    "ollama":     {"url": "http://localhost:11434", "health_path": "/api/tags"},
}

# PostgreSQL
PG_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "database": os.getenv("POSTGRES_DB", "xiaocai"),
    "user": os.getenv("POSTGRES_USER", "xiaocai"),
    "password": os.getenv("POSTGRES_PASSWORD", ""),
}

# Redis
REDIS_CONFIG = {
    "host": os.getenv("REDIS_HOST", "localhost"),
    "port": int(os.getenv("REDIS_PORT", "6379")),
}

# APIPro
APIPRO_BASE_URL = os.getenv("APIPRO_BASE_URL", "https://api.apipro.ai/v1")
APIPRO_API_KEY  = os.getenv("APIPRO_API_KEY", "")

# 输出目录
import json
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
SNAPSHOT_FILE = f"{DATA_DIR}/system_inventory_{datetime.now().strftime('%Y%m%d_%H%M')}.json"

print("配置加载完成")

## 1. 依赖导入

In [ ]:
import httpx
import time
import subprocess
import socket
from datetime import datetime

print("依赖导入完成")

## 2. HTTP 服务探测

In [ ]:
def probe_http_service(name: str, url: str, health_path: str, timeout: float = 5.0) -> dict:
    """探测 HTTP 服务：连通性 + 延迟 + 版本信息"""
    full_url = f"{url}{health_path}"
    result = {
        "service": name,
        "url": url,
        "health_url": full_url,
        "status": "unknown",
        "latency_ms": None,
        "http_status": None,
        "response_body": None,
        "error": None,
    }
    try:
        start = time.perf_counter()
        resp = httpx.get(full_url, timeout=timeout, follow_redirects=True)
        elapsed = (time.perf_counter() - start) * 1000
        result["latency_ms"] = round(elapsed, 1)
        result["http_status"] = resp.status_code
        result["status"] = "ok" if resp.status_code < 400 else "error"
        try:
            result["response_body"] = resp.json()
        except Exception:
            result["response_body"] = resp.text[:200]
    except httpx.ConnectError:
        result["status"] = "unreachable"
        result["error"] = "Connection refused"
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)[:200]
    return result


print("=" * 65)
print("HTTP 服务状态")
print("=" * 65)

http_results = {}
for svc_name, svc_cfg in SERVICES.items():
    r = probe_http_service(svc_name, svc_cfg["url"], svc_cfg["health_path"])
    http_results[svc_name] = r
    icon = "OK  " if r["status"] == "ok" else "FAIL"
    latency = f"{r['latency_ms']:.1f}ms" if r["latency_ms"] else "N/A"
    print(f"[{icon}] {svc_name:15s} {svc_cfg['url']:30s} {latency:10s} HTTP:{r['http_status']}")
    if r["error"]:
        print(f"       错误: {r['error']}")

print("=" * 65)

## 3. LLM Provider 清单

In [ ]:
def probe_apipro_models(api_key: str, base_url: str) -> dict:
    """探测 APIPro 可用模型列表"""
    if not api_key:
        return {"status": "no_key", "models": [], "error": "APIPRO_API_KEY 未配置"}
    try:
        resp = httpx.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=10.0
        )
        if resp.status_code == 200:
            data = resp.json()
            models = [m["id"] for m in data.get("data", [])]
            return {"status": "ok", "model_count": len(models), "models": models[:20]}
        return {"status": "error", "models": [], "error": f"HTTP {resp.status_code}: {resp.text[:200]}"}
    except Exception as e:
        return {"status": "error", "models": [], "error": str(e)}


def probe_ollama_models(base_url: str) -> dict:
    """探测 Ollama 已安装模型"""
    try:
        resp = httpx.get(f"{base_url}/api/tags", timeout=5.0)
        if resp.status_code == 200:
            data = resp.json()
            models = [
                {
                    "name": m["name"],
                    "size_gb": round(m.get("size", 0) / 1e9, 2),
                    "modified": m.get("modified_at", "")[:10],
                }
                for m in data.get("models", [])
            ]
            return {"status": "ok", "model_count": len(models), "models": models}
        return {"status": "error", "models": [], "error": f"HTTP {resp.status_code}"}
    except Exception as e:
        return {"status": "unreachable", "models": [], "error": str(e)}


print("=" * 65)
print("LLM Provider 清单")
print("=" * 65)

# APIPro
apipro_info = probe_apipro_models(APIPRO_API_KEY, APIPRO_BASE_URL)
print(f"\nAPIPro ({APIPRO_BASE_URL})")
print(f"  状态: {apipro_info['status']}")
if apipro_info["status"] == "ok":
    print(f"  可用模型数: {apipro_info['model_count']}")
    # 显示项目使用的关键模型
    target_models = ["claude", "qwen", "deepseek"]
    for m in apipro_info["models"]:
        if any(t in m.lower() for t in target_models):
            print(f"    - {m}")
elif apipro_info["error"]:
    print(f"  错误: {apipro_info['error']}")

# Ollama
ollama_url = SERVICES["ollama"]["url"]
ollama_info = probe_ollama_models(ollama_url)
print(f"\nOllama ({ollama_url})")
print(f"  状态: {ollama_info['status']}")
if ollama_info["status"] == "ok":
    print(f"  已安装模型数: {ollama_info['model_count']}")
    for m in ollama_info["models"]:
        print(f"    - {m['name']:35s} {m['size_gb']} GB  (修改: {m['modified']})")
elif ollama_info.get("error"):
    print(f"  错误: {ollama_info['error']}")

print("=" * 65)

## 4. PostgreSQL 探测

In [ ]:
def probe_postgres(config: dict) -> dict:
    """通过 docker exec 探测 PostgreSQL 状态（无需本地安装 psycopg2）"""
    result = {"status": "unknown", "version": None, "tables": [], "error": None}

    # 尝试 docker exec
    cmds = [
        # 版本
        f"docker exec xiaocai-postgres psql -U {config['user']} -d {config['database']} -t -c 'SELECT version();'",
        # 表列表
        f"docker exec xiaocai-postgres psql -U {config['user']} -d {config['database']} -t -c \"SELECT tablename FROM pg_tables WHERE schemaname='public';\"",
    ]

    try:
        start = time.perf_counter()
        ver_out = subprocess.run(cmds[0], shell=True, capture_output=True, text=True, timeout=5)
        elapsed = (time.perf_counter() - start) * 1000

        if ver_out.returncode == 0:
            result["status"] = "ok"
            result["latency_ms"] = round(elapsed, 1)
            result["version"] = ver_out.stdout.strip().split("\n")[0].strip()

            tbl_out = subprocess.run(cmds[1], shell=True, capture_output=True, text=True, timeout=5)
            if tbl_out.returncode == 0:
                result["tables"] = [t.strip() for t in tbl_out.stdout.strip().split("\n") if t.strip()]
        else:
            result["status"] = "error"
            result["error"] = ver_out.stderr.strip()[:200]
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)

    return result


print("=" * 65)
print("PostgreSQL 状态")
print("=" * 65)

pg_info = probe_postgres(PG_CONFIG)
print(f"状态:   {pg_info['status']}")
if pg_info.get("version"):
    print(f"版本:   {pg_info['version']}")
    print(f"连接延迟: {pg_info.get('latency_ms', 'N/A')} ms")
    print(f"表清单 ({len(pg_info['tables'])} 张):")
    for t in pg_info["tables"]:
        print(f"  - {t}")
elif pg_info.get("error"):
    print(f"错误: {pg_info['error']}")
    print("提示: 检查 docker ps 确认 xiaocai-postgres 容器运行")

print("=" * 65)

## 5. Redis 探测

In [ ]:
def probe_redis(config: dict) -> dict:
    """通过 docker exec 探测 Redis 状态"""
    result = {"status": "unknown", "version": None, "info": {}, "error": None}

    try:
        start = time.perf_counter()
        ping_out = subprocess.run(
            f"docker exec xiaocai-redis redis-cli PING",
            shell=True, capture_output=True, text=True, timeout=5
        )
        elapsed = (time.perf_counter() - start) * 1000

        if ping_out.returncode == 0 and "PONG" in ping_out.stdout:
            result["status"] = "ok"
            result["latency_ms"] = round(elapsed, 1)

            # 获取 INFO
            info_out = subprocess.run(
                "docker exec xiaocai-redis redis-cli INFO server | grep -E 'redis_version|uptime_in_seconds|used_memory_human'",
                shell=True, capture_output=True, text=True, timeout=5
            )
            if info_out.returncode == 0:
                for line in info_out.stdout.strip().split("\n"):
                    if ":" in line:
                        k, v = line.split(":", 1)
                        result["info"][k.strip()] = v.strip()
                result["version"] = result["info"].get("redis_version", "unknown")

            # Key 数量
            dbsize_out = subprocess.run(
                "docker exec xiaocai-redis redis-cli DBSIZE",
                shell=True, capture_output=True, text=True, timeout=3
            )
            if dbsize_out.returncode == 0:
                result["key_count"] = int(dbsize_out.stdout.strip())
        else:
            result["status"] = "error"
            result["error"] = ping_out.stderr.strip() or "PING 失败"
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)

    return result


print("=" * 65)
print("Redis 状态")
print("=" * 65)

redis_info = probe_redis(REDIS_CONFIG)
print(f"状态:     {redis_info['status']}")
if redis_info["status"] == "ok":
    print(f"版本:     {redis_info.get('version', 'N/A')}")
    print(f"PING 延迟: {redis_info.get('latency_ms', 'N/A')} ms")
    print(f"当前 Key 数: {redis_info.get('key_count', 'N/A')}")
    for k, v in redis_info.get("info", {}).items():
        print(f"  {k}: {v}")
elif redis_info.get("error"):
    print(f"错误: {redis_info['error']}")
    print("提示: 检查 docker ps 确认 xiaocai-redis 容器运行")

print("=" * 65)

## 6. Qdrant 详细状态

In [ ]:
def probe_qdrant_detail(url: str) -> dict:
    """探测 Qdrant 版本 + Collections 详情"""
    result = {"status": "unknown", "version": None, "collections": [], "error": None}
    try:
        # 版本
        resp = httpx.get(f"{url}/", timeout=5.0)
        if resp.status_code == 200:
            data = resp.json()
            result["version"] = data.get("version", "unknown")
            result["status"] = "ok"

        # Collections
        colls_resp = httpx.get(f"{url}/collections", timeout=5.0)
        if colls_resp.status_code == 200:
            colls_data = colls_resp.json()
            colls = colls_data.get("result", {}).get("collections", [])

            for coll in colls:
                coll_name = coll["name"]
                # 获取每个 collection 的详细信息
                detail_resp = httpx.get(f"{url}/collections/{coll_name}", timeout=5.0)
                if detail_resp.status_code == 200:
                    detail = detail_resp.json().get("result", {})
                    config = detail.get("config", {})
                    params = config.get("params", {})
                    vectors_config = params.get("vectors", {})

                    # 处理不同的 vector 配置格式
                    if isinstance(vectors_config, dict) and "size" in vectors_config:
                        dim = vectors_config["size"]
                        distance = vectors_config.get("distance", "unknown")
                    else:
                        dim = "?"
                        distance = "?"

                    result["collections"].append({
                        "name": coll_name,
                        "vectors_count": detail.get("vectors_count", 0),
                        "points_count": detail.get("points_count", 0),
                        "dimension": dim,
                        "distance": distance,
                        "status": detail.get("status", "unknown"),
                    })
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)
    return result


print("=" * 65)
print("Qdrant 详细状态")
print("=" * 65)

qdrant_url = SERVICES["qdrant"]["url"]
qdrant_info = probe_qdrant_detail(qdrant_url)
print(f"状态:   {qdrant_info['status']}")
print(f"版本:   {qdrant_info.get('version', 'N/A')}")
print(f"URL:    {qdrant_url}")
if qdrant_info["collections"]:
    print(f"\nCollections ({len(qdrant_info['collections'])} 个):")
    for c in qdrant_info["collections"]:
        print(f"  {c['name']:30s} 向量:{c['vectors_count']:6d}  维度:{c['dimension']}  距离:{c['distance']}  状态:{c['status']}")
else:
    print("  (无 Collections)")

if qdrant_info.get("error"):
    print(f"错误: {qdrant_info['error']}")

print("=" * 65)

## 7. Docker 容器状态

In [ ]:
def probe_docker_containers() -> list:
    """列出所有 xiaocai-* 容器的状态"""
    try:
        out = subprocess.run(
            "docker ps -a --filter name=xiaocai --format '{{.Names}}|{{.Status}}|{{.Image}}|{{.Ports}}'",
            shell=True, capture_output=True, text=True, timeout=10
        )
        containers = []
        for line in out.stdout.strip().split("\n"):
            if "|" in line:
                parts = line.split("|")
                containers.append({
                    "name": parts[0],
                    "status": parts[1],
                    "image": parts[2],
                    "ports": parts[3] if len(parts) > 3 else "",
                })
        return containers
    except Exception as e:
        return [{"name": "error", "status": str(e), "image": "", "ports": ""}]


print("=" * 80)
print("Docker 容器状态 (xiaocai-*)")
print("=" * 80)

containers = probe_docker_containers()
if containers:
    print(f"{'容器名称':35s} {'状态':25s} {'镜像':30s}")
    print("-" * 80)
    for c in containers:
        print(f"{c['name']:35s} {c['status']:25s} {c['image']:30s}")
        if c['ports']:
            print(f"  端口: {c['ports']}")
else:
    print("未找到 xiaocai-* 容器")

print("=" * 80)

## 8. AI Engine 详细状态（/status 端点）

In [ ]:
def probe_ai_engine_status(url: str) -> dict:
    """调用 AI Engine /status 获取 LLM provider 详情"""
    try:
        resp = httpx.get(f"{url}/status", timeout=10.0)
        if resp.status_code == 200:
            return {"status": "ok", "data": resp.json()}
        return {"status": "error", "data": None, "error": f"HTTP {resp.status_code}"}
    except Exception as e:
        return {"status": "unreachable", "data": None, "error": str(e)}


print("=" * 65)
print("AI Engine LLM Provider 状态")
print("=" * 65)

engine_url = SERVICES["ai_engine"]["url"]
engine_status = probe_ai_engine_status(engine_url)

if engine_status["status"] == "ok" and engine_status["data"]:
    data = engine_status["data"]
    print(f"当前活跃 Provider: {data.get('active_provider', 'N/A')}")
    print(f"有可用 Provider: {data.get('has_available_provider', False)}")
    print("\nProvider 列表:")
    for p in data.get("providers", []):
        avail_str = "可用" if p.get("available") else "不可用"
        print(f"  [{avail_str}] {p['name']:30s} ({p.get('type', '?')})")
else:
    print(f"状态: {engine_status['status']}")
    if engine_status.get("error"):
        print(f"错误: {engine_status['error']}")

print("=" * 65)

## 9. 系统清单汇总（保存快照）

In [ ]:
snapshot = {
    "timestamp": datetime.now().isoformat(),
    "tech_stack": {
        "frontend":    {"name": "React",                "repo": "xiaocai-app",     "port": 3000},
        "api_gateway": {"name": "FastAPI + GraphQL",    "repo": "xiaocai-api",     "port": 8001},
        "ai_engine":   {"name": "FastAPI + LangGraph",  "repo": "xiaocai-ai-engine", "port": 8002},
        "llm_primary": {"name": "APIPro",               "models": ["claude-sonnet-4-5", "qwen-plus", "deepseek-chat"]},
        "llm_local":   {"name": "Ollama",               "models": ["qwen2.5:14b", "qwen2.5:7b"]},
        "embedding":   {"name": "sentence-transformers", "model": "all-MiniLM-L6-v2", "dimensions": 384},
        "vector_db":   {"name": "Qdrant",               "port": 6333},
        "cache":       {"name": "Redis",                "port": 6379},
        "database":    {"name": "PostgreSQL",           "port": 5432},
        "proxy":       {"name": "Nginx",                "ports": [80, 443]},
        "orchestration": {"name": "LangGraph"},
    },
    "runtime_status": {
        "http_services": {k: {"status": v["status"], "latency_ms": v["latency_ms"]} for k, v in http_results.items()},
        "postgres": {"status": pg_info["status"], "version": pg_info.get("version"), "tables": pg_info.get("tables", [])},
        "redis": {"status": redis_info["status"], "version": redis_info.get("version"), "key_count": redis_info.get("key_count")},
        "qdrant": {"status": qdrant_info["status"], "version": qdrant_info.get("version"), "collections": qdrant_info.get("collections", [])},
        "apipro": {"status": apipro_info["status"]},
        "ollama": {"status": ollama_info["status"], "installed_models": [m["name"] for m in ollama_info.get("models", [])]},
        "containers": containers,
    }
}

with open(SNAPSHOT_FILE, "w", encoding="utf-8") as f:
    json.dump(snapshot, f, ensure_ascii=False, indent=2)

print("=" * 65)
print("系统清单汇总")
print("=" * 65)
print(f"快照保存: {SNAPSHOT_FILE}")
print()
print("技术栈一览:")
for k, v in snapshot["tech_stack"].items():
    print(f"  {k:15s}: {v['name']}")

print()
print("运行状态:")
all_ok = True
for svc, info in snapshot["runtime_status"]["http_services"].items():
    icon = "OK  " if info["status"] == "ok" else "FAIL"
    lat = f"{info['latency_ms']:.1f}ms" if info["latency_ms"] else "N/A"
    print(f"  [{icon}] {svc:15s} {lat}")
    if info["status"] != "ok":
        all_ok = False

for svc in ["postgres", "redis", "qdrant"]:
    info = snapshot["runtime_status"][svc]
    icon = "OK  " if info["status"] == "ok" else "FAIL"
    print(f"  [{icon}] {svc:15s}")
    if info["status"] != "ok":
        all_ok = False

print()
if all_ok:
    print("所有服务正常，可以运行后续 Notebook (04~08)")
else:
    print("部分服务异常，请先修复后再运行后续 Notebook")

print("=" * 65)